In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, Ridge, LassoCV, Lasso, ElasticNetCV, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

# Load dataset
df = pd.read_csv('regression_housing_dataset.csv')

# Separate independent and dependent variables
X = df.drop('price_usd_k', axis=1)
y = df['price_usd_k']

# Split data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=20
)

# Standardize features for regularized regression models
scaler = StandardScaler()
x_train_scaled = pd.DataFrame(
    scaler.fit_transform(x_train), columns=x_train.columns, index=x_train.index
)
x_test_scaled = pd.DataFrame(
    scaler.transform(x_test), columns=x_test.columns, index=x_test.index
)

# Tables to store model coefficients and performance
coef_table = pd.DataFrame({'Feature': X.columns})
results = []

#OLS Multiple Regression

In [ ]:
# Add an intercept for OLS regression
x_train_ols = sm.add_constant(x_train, has_constant='add')

# Fit OLS model using original, unscaled features
mod1 = sm.OLS(y_train, x_train_ols).fit()

# Display regression summary
print(mod1.summary())

# Predict values for the test dataset
x_test_ols = sm.add_constant(x_test, has_constant='add')
y_pred_ols = mod1.predict(x_test_ols)

# Evaluate OLS model
results.append({
    'Model': 'OLS',
    'R2 Score': r2_score(y_test, y_pred_ols),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_ols))
})

# Store OLS coefficients
coef_table['Coeff_OLS'] = [
    mod1.params[feature] for feature in X.columns
]

                            OLS Regression Results                            
Dep. Variable:            price_usd_k   R-squared:                       0.956
Model:                            OLS   Adj. R-squared:                  0.955
Method:                 Least Squares   F-statistic:                     1696.
Date:                Tue, 20 Oct 2026   Prob (F-statistic):               0.00
Time:                        18:56:13   Log-Likelihood:                -3192.8
No. Observations:                 800   AIC:                             6408.
Df Residuals:                     789   BIC:                             6459.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const               38.7092      3.649  


## Hypothesis Testing: OLS Multiple Regression

**Test used:** t-test on individual regression coefficients (βᵢ)

- **Null hypothesis (H₀):** βᵢ = 0, the predictor has no significant linear relationship with house price, holding other predictors constant.
- **Alternative hypothesis (H₁):** βᵢ ≠ 0, the predictor has a significant linear relationship with house price, holding other predictors constant.
- **Significance level:** α = 0.05

### Results

| **Coefficient** | **Estimate** | **t-statistic** | **p-value** | **95% CI** |
|---|---:|---:|---:|---|
| square_feet | 0.1021 | 3.084 | 0.002 | [0.037, 0.167] |
| bedrooms | 13.3860 | 6.437 | 0.000 | [9.304, 17.468] |
| bathrooms | 17.3446 | 7.294 | 0.000 | [12.677, 22.012] |
| age_years | -2.1509 | -50.554 | 0.000 | [-2.234, -2.067] |
| dist_city_miles | -27.8065 | -0.192 | 0.847 | [-311.513, 255.900] |
| living_area_sqft | 0.0062 | 0.187 | 0.852 | [-0.059, 0.071] |
| dist_city_km | 15.1475 | 0.169 | 0.866 | [-161.137, 191.432] |
| garage_sqft | -0.0015 | -0.140 | 0.889 | [-0.023, 0.020] |
| random_index_1 | 0.0577 | 1.259 | 0.208 | [-0.032, 0.148] |
| random_index_2 | -0.0132 | -0.806 | 0.421 | [-0.045, 0.019] |

### Decision

- **square_feet:** p-value = 0.002 < 0.05; reject H₀.
- **bedrooms:** p-value < 0.05; reject H₀.
- **bathrooms:** p-value < 0.05; reject H₀.
- **age_years:** p-value < 0.05; reject H₀.
- **dist_city_miles:** p-value = 0.847 > 0.05; fail to reject H₀.
- **living_area_sqft:** p-value = 0.852 > 0.05; fail to reject H₀.
- **dist_city_km:** p-value = 0.866 > 0.05; fail to reject H₀.
- **garage_sqft:** p-value = 0.889 > 0.05; fail to reject H₀.
- **random_index_1:** p-value = 0.208 > 0.05; fail to reject H₀.
- **random_index_2:** p-value = 0.421 > 0.05; fail to reject H₀.

### Final Conclusion

At the 5% significance level, **square_feet, bedrooms, bathrooms, and age_years are statistically significant predictors** of house price when controlling for the other variables in the model.

The coefficients for square_feet, bedrooms, and bathrooms are positive, indicating positive conditional associations with house price. The coefficient for age_years is negative, indicating that greater property age is associated with a lower predicted price, holding other predictors constant.


The remaining six predictors are not statistically significant at the 5% level because their p-values exceed 0.05. This does not prove that they have no effect; the model does not provide sufficient evidence of a nonzero coefficient for these predictors.

**Overall model test:** F-statistic = 1696, with Prob (F-statistic) ≈ 0.00. Since the overall p-value is below 0.05, reject the null hypothesis that all ten slope coefficients are simultaneously zero. The model is statistically significant overall, with R² = 0.956, meaning it explains approximately 95.6% of the variation in house price in the training data.


#Ridge regression

In [ ]:
# Select the best regularization parameter using cross-validation
ridgecv = RidgeCV(cv=5)
ridgecv.fit(x_train_scaled, y_train)

# Fit Ridge using the selected alpha
ridge = Ridge(alpha=ridgecv.alpha_)
ridge.fit(x_train_scaled, y_train)

print("Best Ridge Alpha:", ridgecv.alpha_)
print("Ridge Coefficients:\n", pd.Series(ridge.coef_, index=x_train.columns))

# Evaluate Ridge on test data
y_pred_ridge = ridge.predict(x_test_scaled)

results.append({
    'Model': 'Ridge',
    'R2 Score': r2_score(y_test, y_pred_ridge),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_ridge))
})

# Store Ridge coefficients
coef_table['Coeff_Ridge'] = ridge.coef_

Best Ridge Alpha: 1.0
Ridge Coefficients:
 square_feet         27.586089
bedrooms            14.400687
bathrooms           13.909193
age_years          -23.815928
dist_city_miles    -11.981696
living_area_sqft    14.760488
dist_city_km       -11.953392
garage_sqft         -0.153435
random_index_1       0.575266
random_index_2      -0.375192
dtype: float64


#Lasso Regression

In [ ]:
# Select the best alpha using cross-validation
lassoCV = LassoCV(cv=5)
lassoCV.fit(x_train_scaled, y_train)

# Fit Lasso using the selected alpha
lasso = Lasso(alpha=lassoCV.alpha_)
lasso.fit(x_train_scaled, y_train)

print("Best Lasso Alpha:", lassoCV.alpha_)
print("Lasso Coefficients:\n", pd.Series(lasso.coef_, index=x_train.columns))

# Evaluate Lasso on test data
y_pred_lasso = lasso.predict(x_test_scaled)

results.append({
    'Model': 'Lasso',
    'R2 Score': r2_score(y_test, y_pred_lasso),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_lasso))
})

# Store Lasso coefficients
coef_table['Coeff_Lasso'] = lasso.coef_

Best Lasso Alpha: 0.08852987644094329
Lasso Coefficients:
 square_feet         40.924930
bedrooms            14.452433
bathrooms           13.630747
age_years          -23.778542
dist_city_miles    -23.867193
living_area_sqft     1.364877
dist_city_km        -0.000000
garage_sqft          0.000000
random_index_1       0.506075
random_index_2      -0.295210
dtype: float64


#Elastic Net regression

In [ ]:
# Select alpha and l1_ratio using cross-validation
elasticCV = ElasticNetCV(cv=5)
elasticCV.fit(x_train_scaled, y_train)

# Fit Elastic Net using the selected parameters
elastic = ElasticNet(
    alpha=elasticCV.alpha_,
    l1_ratio=elasticCV.l1_ratio_
)
elastic.fit(x_train_scaled, y_train)

print("Best Elastic Net Alpha:", elasticCV.alpha_)
print("Best L1 Ratio:", elasticCV.l1_ratio_)
print("Elastic Net Coefficients:\n", pd.Series(elastic.coef_, index=x_train.columns))

# Evaluate Elastic Net on test data
y_pred_elastic = elastic.predict(x_test_scaled)

results.append({
    'Model': 'Elastic Net',
    'R2 Score': r2_score(y_test, y_pred_elastic),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_elastic))
})

# Store Elastic Net coefficients
coef_table['Coeff_Elastic'] = elastic.coef_

Best Elastic Net Alpha: 0.08218385718588032
Best L1 Ratio: 0.5
Elastic Net Coefficients:
 square_feet         21.044618
bedrooms            12.190197
bathrooms           12.874833
age_years          -22.777200
dist_city_miles    -11.752472
living_area_sqft    20.374075
dist_city_km       -11.713741
garage_sqft          2.740613
random_index_1       0.499546
random_index_2      -0.343037
dtype: float64


In [ ]:
# Display the coefficients of every model side by side
coefficients

,Feature,OLS,Ridge,Lasso,Elastic Net
0,square_feet,39.958188,24.612939,40.924930,21.044618
1,bedrooms,14.554450,14.224488,14.452433,12.190197
2,bathrooms,13.799325,13.915319,13.630747,12.874833
3,age_years,-23.868536,-23.768544,-23.778542,-22.777200
4,dist_city_miles,-194.208534,-11.964673,-23.867193,-11.752472
5,living_area_sqft,2.418865,17.695396,1.364877,20.374075
6,dist_city_km,170.261634,-11.952340,-0.000000,-11.713741
7,garage_sqft,-0.190395,0.006895,0.000000,2.740613
8,random_index_1,0.589444,0.569635,0.506075,0.499546
9,random_index_2,-0.376499,-0.374158,-0.295210,-0.343037


In [ ]:
# Compare predictive performance on the same test dataset
performance_table = pd.DataFrame(results)

performance_table.sort_values(
    by='RMSE', ascending=True
).reset_index(drop=True)

,Model,R2 Score,RMSE
0,Ridge,0.965015,11.956751
1,Ridge,0.965015,11.956751
2,Ridge,0.965015,11.956751
3,Ridge,0.965003,11.958797
4,OLS,0.964860,11.983239
5,Lasso,0.964810,11.991751
6,Lasso,0.964810,11.991751
7,Lasso,0.964810,11.991751
8,Lasso,0.964810,11.991751
9,Lasso,0.964810,11.991751
